IPL CASE STUDY

SETUP: Initial Setup

In [51]:
import pandas as pd
import numpy as np

print(f"Pandas Version: {pd.__version__}")

Pandas Version: 3.0.2


Stage 1: Data Ingestion

In [58]:
deliveries_df = pd.read_csv("deliveries.csv")
matches_df = pd.read_csv("matches.csv")


deliveries_df.shape
deliveries_df.columns
deliveries_df.dtypes

matches_df.shape
matches_df.columns
matches_df.dtypes

id                   int64
season                 str
city                   str
date                   str
match_type             str
player_of_match        str
venue                  str
team1                  str
team2                  str
toss_winner            str
toss_decision          str
winner                 str
result                 str
result_margin      float64
target_runs        float64
target_overs       float64
super_over             str
method                 str
umpire1                str
umpire2                str
dtype: object

Stage 2: Data Cleaning & Validation

In [56]:
deliveries_df.isna().sum()
matches_df.isna().sum()

deliveries_df["match_id"].nunique()
matches_df["id"].nunique()

unmatched_ids = set(deliveries_df["match_id"]) - set(matches_df["id"])
len(unmatched_ids) #0

deliveries_df["inning"].unique()
deliveries_df["over"].min(), deliveries_df["over"].max()

(deliveries_df["batsman_runs"] + deliveries_df["extra_runs"] - deliveries_df["total_runs"]).value_counts()

0    260920
Name: count, dtype: int64

Stage 3: Data Transformation

In [70]:
deliveries_df["calculated_total_runs"] = (
    deliveries_df["batsman_runs"] + deliveries_df["extra_runs"]
)
(
    deliveries_df["total_runs"]
    - deliveries_df["calculated_total_runs"]
).value_counts()

matches_df = matches_df.rename(columns={"id": "match_id"})

matches_subset = matches_df[
    [
        "match_id",
        "season",
        "city",
        "venue",
        "team1",
        "team2",
        "toss_winner",
        "toss_decision",
        "winner",
        "player_of_match"
    ]
]

ipl_df = deliveries_df.merge(
    matches_subset,
    on="match_id",
    how="inner"
)

ipl_df.shape
ipl_df.columns
ipl_df.isna().sum()

match_id                      0
inning                        0
batting_team                  0
bowling_team                  0
over                          0
ball                          0
batter                        0
bowler                        0
non_striker                   0
batsman_runs                  0
extra_runs                    0
total_runs                    0
extras_type              246795
is_wicket                     0
player_dismissed         247970
dismissal_kind           247970
fielder                  251566
calculated_total_runs         0
season                        0
city                      12397
venue                         0
team1                         0
team2                         0
toss_winner                   0
toss_decision                 0
winner                      490
player_of_match             490
dtype: int64

CORE ANALYSIS

Total Runs Per Match

In [77]:
runs_per_match = (
    ipl_df
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index()
    .rename(columns={"total_runs": "total_runs_per_match"})
)

runs_per_match.head(10)

,match_id,total_runs_per_match
0,335982,304
1,335983,447
2,335984,261
3,335985,331
4,335986,222
5,335987,334
6,335988,285
7,335989,410
8,335990,431
9,335991,298


Runs Per Team Per Match

In [79]:
team_runs_per_match = (
    ipl_df
    .groupby(["match_id", "batting_team"])["total_runs"]
    .sum()
    .reset_index()
    .rename(columns={"batting_team": "team", "total_runs": "team_runs_per_match"})
)

team_runs_per_match.head(10)


,match_id,team,team_runs_per_match
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Kings XI Punjab,207
4,335984,Delhi Daredevils,132
5,335984,Rajasthan Royals,129
6,335985,Mumbai Indians,165
7,335985,Royal Challengers Bangalore,166
8,335986,Deccan Chargers,110
9,335986,Kolkata Knight Riders,112


Top 10 Batters

In [83]:
batter_total_runs = (
    ipl_df
    .groupby("batter")["batsman_runs"]
    .sum()
    .reset_index()
    .rename(columns={"batsman_runs": "total_runs_by_batter"})
    .sort_values(by="total_runs_by_batter", ascending=False)
)

batter_total_runs.head(10)

,batter,total_runs_by_batter
631,V Kohli,8014
512,S Dhawan,6769
477,RG Sharma,6630
147,DA Warner,6567
546,SK Raina,5536
374,MS Dhoni,5243
30,AB de Villiers,5181
124,CH Gayle,4997
501,RV Uthappa,4954
282,KD Karthik,4843


Strike Rate of Batters 

In [90]:
valid_balls_df = ipl_df[
    (ipl_df["extras_type"].isna()) |
    (ipl_df["extras_type"] != "wides")
]

runs_per_batter = (
    valid_balls_df
    .groupby("batter")["batsman_runs"]
    .sum()
    .reset_index(name="runs")
)

balls_faced_per_batter = (
    valid_balls_df
    .groupby("batter")
    .size()
    .reset_index(name="balls_faced")
)

batting_sr_df = runs_per_batter.merge(
    balls_faced_per_batter,
    on="batter",
    how="inner"
)

batting_sr_df["strike_rate"] = (
    batting_sr_df["runs"] / batting_sr_df["balls_faced"]
) * 100

batting_sr_df = batting_sr_df[
    batting_sr_df["balls_faced"] >= 100
]

batting_sr_df.sort_values(by="strike_rate", ascending=False).head(10)


,batter,runs,balls_faced,strike_rate
234,J Fraser-McGurk,330,141,234.042553
652,WG Jacks,230,131,175.572519
433,PD Salt,653,372,175.537634
39,AD Russell,2488,1423,174.841883
617,TM Head,772,444,173.873874
606,T Stubbs,405,233,173.819742
612,TH David,659,387,170.284238
105,BCJ Cutting,238,141,168.794326
208,H Klaasen,993,590,168.305085
85,Ashutosh Sharma,189,113,167.256637


Top 10 Bowlers by Economy

In [93]:
bowler_runs_df = ipl_df[
    ~ipl_df["extras_type"].isin(["byes", "legbyes"])
]

legal_deliveries_df = ipl_df[
    ~ipl_df["extras_type"].isin(["wides", "noballs"])
]

runs_conceded = (
    bowler_runs_df
    .groupby("bowler")["total_runs"]
    .sum()
    .reset_index(name="runs_conceded")
)

balls_bowled = (
    legal_deliveries_df
    .groupby("bowler")
    .size()
    .reset_index(name="balls_bowled")
)

bowler_stats = runs_conceded.merge(
    balls_bowled,
    on="bowler",
    how="inner"
)

bowler_stats["overs"] = bowler_stats["balls_bowled"] / 6
bowler_stats["economy"] = (
    bowler_stats["runs_conceded"] / bowler_stats["overs"]
)

bowler_stats = bowler_stats[
    bowler_stats["overs"] >= 50
]

top_10_economy_bowlers = (
    bowler_stats
    .sort_values(by="economy", ascending=True)
    .head(10)
)

top_10_economy_bowlers


,bowler,runs_conceded,balls_bowled,overs,economy
7,A Kumble,1058,965,160.833333,6.578238
147,GD McGrath,357,324,54.000000,6.611111
263,M Muralitharan,1706,1528,254.666667,6.698953
446,SP Narine,4582,4081,680.166667,6.736584
377,RE van der Merwe,498,443,73.833333,6.744921
130,DL Vettori,879,777,129.500000,6.787645
398,Rashid Khan,3267,2872,478.666667,6.825209
181,J Yadav,445,390,65.000000,6.846154
176,J Botha,800,694,115.666667,6.916427
284,MJ Santner,422,366,61.000000,6.918033


Most Consistent Batters

In [94]:
batter_match_runs = (
    ipl_df
    .groupby(["batter", "match_id"])["batsman_runs"]
    .sum()
    .reset_index(name="runs_in_match")
)

batter_consistency = (
    batter_match_runs
    .groupby("batter")
    .agg(
        matches_played=("match_id", "nunique"),
        average_runs_per_match=("runs_in_match", "mean")
    )
    .reset_index()
)

consistent_batters = batter_consistency[
    batter_consistency["matches_played"] >= 50
]

most_consistent_batters = (
    consistent_batters
    .sort_values(by="average_runs_per_match", ascending=False)
    .head(10)
)
most_consistent_batters

,batter,matches_played,average_runs_per_match
289,KL Rahul,122,38.434426
473,RD Gaikwad,65,36.615385
542,SE Marsh,69,36.072464
147,DA Warner,184,35.690217
124,CH Gayle,141,35.439716
352,MEK Hussey,58,34.086207
242,JC Buttler,106,33.801887
188,F du Plessis,138,33.123188
631,V Kohli,244,32.844262
592,Shubman Gill,99,32.484848


Highest Individual Score in a Match

In [97]:
batter_match_scores = (
    ipl_df
    .groupby(["match_id", "batter"])["batsman_runs"]
    .sum()
    .reset_index(name="runs_scored")
)

highest_individual_scores = (
    batter_match_scores
    .sort_values(by="runs_scored", ascending=False)
    .head(10)
)

highest_individual_scores


,match_id,batter,runs_scored
5302,598027,CH Gayle,175
2,335982,BB McCullum,158
14108,1304112,Q de Kock,140
7528,829795,AB de Villiers,133
11583,1216510,KL Rahul,132
15383,1370352,Shubman Gill,129
8359,980987,AB de Villiers,129
4687,548372,CH Gayle,128
10149,1136602,RR Pant,128
2237,419137,M Vijay,127


Boundary Analysis

In [98]:
boundary_df = ipl_df[
    ipl_df["batsman_runs"].isin([4, 6])
]

boundary_counts = (
    boundary_df["batsman_runs"]
    .value_counts()
    .rename_axis("boundary_type")
    .reset_index(name="count")
)

batter_boundaries = (
    boundary_df
    .groupby("batter")["batsman_runs"]
    .count()
    .reset_index(name="boundary_count")
)

top_boundary_players = (
    batter_boundaries
    .sort_values(by="boundary_count", ascending=False)
    .head(10)
)

top_boundary_players

,batter,boundary_count
532,V Kohli,981
437,S Dhawan,921
126,DA Warner,899
409,RG Sharma,880
109,CH Gayle,767
462,SK Raina,710
25,AB de Villiers,667
427,RV Uthappa,663
243,KD Karthik,627
323,MS Dhoni,615


Boundary Percentage

In [100]:
total_runs_per_batter = (
    ipl_df
    .groupby("batter")["batsman_runs"]
    .sum()
    .reset_index(name="total_runs")
)

boundary_runs_df = ipl_df[
    ipl_df["batsman_runs"].isin([4, 6])
]

boundary_runs_per_batter = (
    boundary_runs_df
    .groupby("batter")["batsman_runs"]
    .sum()
    .reset_index(name="boundary_runs")
)

boundary_percentage_df = total_runs_per_batter.merge(
    boundary_runs_per_batter,
    on="batter",
    how="left"
)

boundary_percentage_df["boundary_runs"] = (
    boundary_percentage_df["boundary_runs"].fillna(0)
)

boundary_percentage_df["boundary_percentage"] = (
    boundary_percentage_df["boundary_runs"]
    / boundary_percentage_df["total_runs"]
) * 100

boundary_percentage_df = boundary_percentage_df[
    boundary_percentage_df["total_runs"] >= 500
]

top_boundary_percentage_batters = (
    boundary_percentage_df
    .sort_values(by="boundary_percentage", ascending=False)
    .head(10)
)

top_boundary_percentage_batters

,batter,total_runs,boundary_runs,boundary_percentage
561,SP Narine,1534,1238.0,80.704042
39,AD Russell,2488,1938.0,77.893891
433,PD Salt,653,500.0,76.569678
124,CH Gayle,4997,3786.0,75.765459
570,ST Jayasuriya,768,570.0,74.218750
427,P Simran Singh,756,558.0,73.809524
662,YBK Jaiswal,1607,1180.0,73.428749
32,AC Gilchrist,2069,1508.0,72.885452
633,V Sehwag,2728,1972.0,72.287390
431,PC Valthaty,505,364.0,72.079208


Dot Ball Analysis

In [110]:
dot_balls_df = ipl_df[
    ipl_df["total_runs"] == 0
]

total_dot_balls = dot_balls_df.shape[0]

bowler_dot_balls = (
    dot_balls_df
    .groupby("bowler")
    .size()
    .reset_index(name="dot_balls")
)

top_dot_ball_bowlers = (
    bowler_dot_balls
    .sort_values(by="dot_balls", ascending=False)
    .head(10)
)

top_dot_ball_bowlers

,bowler,dot_balls
70,B Kumar,1632
436,SP Narine,1569
348,R Ashwin,1552
341,PP Chawla,1325
159,Harbhajan Singh,1263
188,JJ Bumrah,1228
366,RA Jadeja,1216
511,YS Chahal,1194
482,UT Yadav,1186
8,A Mishra,1185


Runs Per Over Analysis

In [111]:
over_match_runs = (
    ipl_df
    .groupby(["match_id", "over"])["total_runs"]
    .sum()
    .reset_index(name="runs_in_over")
)

average_runs_per_over = (
    over_match_runs
    .groupby("over")["runs_in_over"]
    .mean()
    .reset_index(name="average_runs")
)

average_runs_per_over_sorted = (
    average_runs_per_over
    .sort_values(by="average_runs", ascending=False)
    .head(10)
)
average_runs_per_over_sorted

,over,average_runs
17,17,18.245336
18,18,17.979265
16,16,17.669145
15,15,17.056325
19,19,16.967526
4,4,16.954338
5,5,16.936015
3,3,16.817352
14,14,16.727189
2,2,16.342466


Powerplay Performance

In [112]:
powerplay_df = ipl_df[
    ipl_df["over"].between(1, 6)
]

powerplay_runs_per_match = (
    powerplay_df
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="powerplay_runs")
)

powerplay_team_runs = (
    powerplay_df
    .groupby(["match_id", "batting_team"])["total_runs"]
    .sum()
    .reset_index(name="powerplay_runs")
)

average_powerplay_runs = (
    powerplay_team_runs
    .groupby("batting_team")["powerplay_runs"]
    .mean()
    .reset_index(name="avg_powerplay_runs")
)

top_powerplay_teams = (
    average_powerplay_runs
    .sort_values(by="avg_powerplay_runs", ascending=False)
    .head(10)
)

top_powerplay_teams

,batting_team,avg_powerplay_runs
17,Royal Challengers Bengaluru,57.933333
4,Gujarat Lions,53.633333
2,Delhi Capitals,51.714286
12,Punjab Kings,51.517857
18,Sunrisers Hyderabad,49.708791
5,Gujarat Titans,49.244444
14,Rising Pune Supergiant,49.125000
9,Lucknow Super Giants,48.636364
0,Chennai Super Kings,48.012658
15,Rising Pune Supergiants,48.000000


Death Overs Performance

In [114]:
death_overs_df = ipl_df[
    ipl_df["over"].between(16, 20)
]

death_over_runs_per_match = (
    death_overs_df
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="death_over_runs")
)

death_over_team_runs = (
    death_overs_df
    .groupby(["match_id", "batting_team"])["total_runs"]
    .sum()
    .reset_index(name="death_over_runs")
)

average_death_over_runs = (
    death_over_team_runs
    .groupby("batting_team")["death_over_runs"]
    .mean()
    .reset_index(name="avg_death_over_runs")
)

top_death_over_teams = (
    average_death_over_runs
    .sort_values(by="avg_death_over_runs", ascending=False)
    .head(10)
)

death_over_batter_runs = (
    death_overs_df
    .groupby("batter")["batsman_runs"]
    .sum()
    .reset_index(name="death_over_runs")
)

death_over_batter_runs = death_over_batter_runs[
    death_over_batter_runs["death_over_runs"] >= 200
]

top_death_over_batters = (
    death_over_batter_runs
    .sort_values(by="death_over_runs", ascending=False)
    .head(10)
)
top_death_over_teams
top_death_over_batters

,batter,death_over_runs
328,MS Dhoni,2786
242,KA Pollard,1708
247,KD Karthik,1565
24,AB de Villiers,1421
415,RA Jadeja,1420
421,RG Sharma,1176
182,HH Pandya,1126
550,V Kohli,1099
31,AD Russell,1065
124,DA Miller,988


Run Distribution per Inning

In [115]:
inning_runs_df = (
    ipl_df
    .groupby(["match_id", "inning"])["total_runs"]
    .sum()
    .reset_index(name="inning_runs")
)

inning_summary = (
    inning_runs_df
    .groupby("inning")["inning_runs"]
    .agg(
        total_runs="sum",
        average_runs="mean",
        matches_played="count"
    )
    .reset_index()
)

inning_summary = inning_summary[
    inning_summary["inning"].isin([1, 2])
]

inning_summary


,inning,total_runs,average_runs,matches_played
0,1,181274,165.547032,1095
1,2,166196,152.194139,1092


Toss Impact Analysis

In [116]:
match_inning_team_runs = (
    ipl_df
    .groupby(["match_id", "inning", "batting_team"])["total_runs"]
    .sum()
    .reset_index(name="runs")
)

toss_info = (
    ipl_df[["match_id", "toss_winner"]]
    .drop_duplicates()
)

match_inning_team_runs = match_inning_team_runs.merge(
    toss_info,
    on="match_id",
    how="left"
)

match_inning_team_runs["team_type"] = match_inning_team_runs.apply(
    lambda row: "toss_winner"
    if row["batting_team"] == row["toss_winner"]
    else "opponent",
    axis=1
)

toss_impact_summary = (
    match_inning_team_runs
    .groupby("team_type")["runs"]
    .mean()
    .reset_index(name="average_runs")
)
toss_impact_summary 

,team_type,average_runs
0,opponent,159.169522
1,toss_winner,154.546029


Player of Match Contribution

In [118]:
batter_match_runs = (
    ipl_df
    .groupby(["match_id", "batter"])["batsman_runs"]
    .sum()
    .reset_index(name="runs_scored")
)

top_scorer_per_match = (
    batter_match_runs
    .sort_values(by=["match_id", "runs_scored"], ascending=[True, False])
    .groupby("match_id")
    .first()
    .reset_index()
)

player_of_match_df = (
    ipl_df[["match_id", "player_of_match"]]
    .drop_duplicates()
)

player_match_comparison = top_scorer_per_match.merge(
    player_of_match_df,
    on="match_id",
    how="left"
)

player_match_comparison["is_top_scorer"] = (
    player_match_comparison["batter"]
    == player_match_comparison["player_of_match"]
)

player_of_match_summary = (
    player_match_comparison["is_top_scorer"]
    .value_counts()
    .reset_index(name="count")
    .rename(columns={"index": "player_of_match_is_top_scorer"})
)

player_of_match_summary


,is_top_scorer,count
0,False,599
1,True,496


Venue Wise Analysis

In [122]:
#cleaning the csv

sorted(ipl_df["venue"].unique())

ipl_df["venue_clean"] = (
    ipl_df["venue"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r",.*$", "", regex=True)
)

venue_mapping = {
    # Delhi
    "Feroz Shah Kotla": "Arun Jaitley Stadium",
    "Arun Jaitley Stadium": "Arun Jaitley Stadium",

    # Bengaluru
    "M Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",

    # Chennai
    "MA Chidambaram Stadium": "MA Chidambaram Stadium",

    # Hyderabad
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Stadium",

    # Mohali / Punjab
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association Stadium": "Punjab Cricket Association IS Bindra Stadium",

    # Mumbai
    "Brabourne Stadium": "Brabourne Stadium",
    "Wankhede Stadium": "Wankhede Stadium",

    # Vizag
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium":
        "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",

    # Dharamsala
    "Himachal Pradesh Cricket Association Stadium":
        "Himachal Pradesh Cricket Association Stadium",

    # Jaipur
    "Sawai Mansingh Stadium": "Sawai Mansingh Stadium",

    # Motera (old name)
    "Sardar Patel Stadium": "Narendra Modi Stadium",

    # Abu Dhabi
    "Zayed Cricket Stadium": "Sheikh Zayed Stadium",
}

ipl_df["venue_clean"] = ipl_df["venue_clean"].replace(venue_mapping)

sorted(ipl_df["venue_clean"].unique())


['Arun Jaitley Stadium',
 'Barabati Stadium',
 'Barsapara Cricket Stadium',
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium',
 'Brabourne Stadium',
 'Buffalo Park',
 'De Beers Diamond Oval',
 'Dr DY Patil Sports Academy',
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',
 'Dubai International Cricket Stadium',
 'Eden Gardens',
 'Green Park',
 'Himachal Pradesh Cricket Association Stadium',
 'Holkar Cricket Stadium',
 'JSCA International Stadium Complex',
 'Kingsmead',
 'M Chinnaswamy Stadium',
 'MA Chidambaram Stadium',
 'Maharaja Yadavindra Singh International Cricket Stadium',
 'Maharashtra Cricket Association Stadium',
 'Narendra Modi Stadium',
 'Nehru Stadium',
 'New Wanderers Stadium',
 'Newlands',
 'OUTsurance Oval',
 'Punjab Cricket Association IS Bindra Stadium',
 'Rajiv Gandhi International Stadium',
 'Saurashtra Cricket Association Stadium',
 'Sawai Mansingh Stadium',
 'Shaheed Veer Narayan Singh International Stadium',
 'Sharjah Cricket Stadium',
 'Sheik

In [123]:
matches_per_venue = (
    ipl_df
    .groupby("venue_clean")["match_id"]
    .nunique()
    .reset_index(name="total_matches")
)

runs_per_match = (
    ipl_df
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="match_runs")
)

match_venue_df = (
    ipl_df[["match_id", "venue_clean"]]
    .drop_duplicates()
)

match_runs_with_venue = runs_per_match.merge(
    match_venue_df,
    on="match_id",
    how="left"
)

average_runs_per_venue = (
    match_runs_with_venue
    .groupby("venue_clean")["match_runs"]
    .mean()
    .reset_index(name="average_match_runs")
)

venue_analysis = matches_per_venue.merge(
    average_runs_per_venue,
    on="venue_clean",
    how="inner"
)

venue_analysis.sort_values(
    by="total_matches",
    ascending=False
)

venue_analysis.sort_values(
    by="average_match_runs",
    ascending=False
)

venue_analysis

,venue_clean,total_matches,average_match_runs
0,Arun Jaitley Stadium,90,322.833333
1,Barabati Stadium,7,325.428571
2,Barsapara Cricket Stadium,3,339.666667
3,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,14,306.714286
4,Brabourne Stadium,27,344.629630
5,Buffalo Park,3,266.333333
6,De Beers Diamond Oval,3,299.000000
7,Dr DY Patil Sports Academy,37,307.648649
8,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,15,303.066667
9,Dubai International Cricket Stadium,46,314.130435


City Wise Scoring Trends

In [126]:
runs_per_match = (
    ipl_df
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="match_runs")
)

match_city_df = (
    ipl_df[["match_id", "city"]]
    .drop_duplicates()
)

match_runs_with_city = runs_per_match.merge(
    match_city_df,
    on="match_id",
    how="left"
)

match_runs_with_city = match_runs_with_city.dropna(subset=["city"])

city_run_trends = (
    match_runs_with_city
    .groupby("city")["match_runs"]
    .mean()
    .reset_index(name="average_match_runs")
)

city_run_trends.sort_values(
    by="average_match_runs",
    ascending=False
)

city_run_trends.sort_values(
    by="average_match_runs",
    ascending=False
)

city_run_trends

,city,average_match_runs
0,Abu Dhabi,306.270270
1,Ahmedabad,331.027778
2,Bangalore,311.738462
3,Bengaluru,360.310345
4,Bloemfontein,264.500000
5,Cape Town,256.714286
6,Centurion,304.416667
7,Chandigarh,325.639344
8,Chennai,314.188235
9,Cuttack,325.428571


Season Wise Run Trends

In [127]:
runs_per_match = (
    ipl_df
    .groupby("match_id")["total_runs"]
    .sum()
    .reset_index(name="match_runs")
)

match_season_df = (
    ipl_df[["match_id", "season"]]
    .drop_duplicates()
)

match_runs_with_season = runs_per_match.merge(
    match_season_df,
    on="match_id",
    how="left"
)

season_total_runs = (
    match_runs_with_season
    .groupby("season")["match_runs"]
    .sum()
    .reset_index(name="total_runs")
)

season_average_runs = (
    match_runs_with_season
    .groupby("season")["match_runs"]
    .mean()
    .reset_index(name="average_runs_per_match")
)

season_run_trends = season_total_runs.merge(
    season_average_runs,
    on="season",
    how="inner"
)

season_run_trends = season_run_trends.sort_values(by="season")

season_run_trends

,season,total_runs,average_runs_per_match
0,2007/08,17937,309.258621
1,2009,16353,286.894737
2,2009/10,18883,314.716667
3,2011,21154,289.780822
4,2012,22453,303.418919
5,2013,22602,297.394737
6,2014,18931,315.516667
7,2015,18353,311.067797
8,2016,18862,314.366667
9,2017,18786,318.406780


Winning Team Analysis

In [130]:
team_runs_per_match = (
    ipl_df
    .groupby(["match_id", "batting_team"])["total_runs"]
    .sum()
    .reset_index(name="team_runs")
)

calculated_winner = (
    team_runs_per_match
    .sort_values(by=["match_id", "team_runs"], ascending=[True, False])
    .groupby("match_id")
    .first()
    .reset_index()
    .rename(columns={"batting_team": "calculated_winner"})
)

actual_winner = (
    ipl_df[["match_id", "winner"]]
    .drop_duplicates()
)

winner_comparison = calculated_winner.merge(
    actual_winner,
    on="match_id",
    how="left"
)

winner_comparison["winner_matches"] = (
    winner_comparison["calculated_winner"]
    == winner_comparison["winner"]
)

winner_match_summary = (
    winner_comparison["winner_matches"]
    .value_counts()
    .reset_index(name="count")
    .rename(columns={"index": "is_match_correct"})
)
winner_comparison.head(10)
winner_match_summary


,winner_matches,count
0,True,1076
1,False,19


Stage 5: Derived Insights


In [ ]:
most_consistent_batters = (
    most_consistent_batters
    .sort_values(by="average_runs_per_match", ascending=False)
)

best_powerplay_teams = (
    top_powerplay_teams
    .sort_values(by="avg_powerplay_runs", ascending=False) 
)

best_death_over_teams = (
    top_death_over_teams
    .sort_values(by="avg_death_over_runs", ascending=False)
)

best_death_over_batters = (
    top_death_over_batters
    .sort_values(by="death_over_runs", ascending=False)
)

bowler_pressure_summary = (
    top_10_economy_bowlers
    .merge(
        top_dot_ball_bowlers,
        on="bowler",
        how="left"
    )
    .fillna(0)
)

high_scoring_venues = (
    venue_analysis
    .sort_values(by="average_match_runs", ascending=False)
)

high_scoring_cities = (
    city_run_trends
    .sort_values(by="average_match_runs", ascending=False)
)

season_trend = (
    season_run_trends
    .sort_values(by="season")
)

toss_impact_insight = toss_impact_summary

win_accuracy = (
    winner_match_summary
    .assign(
        percentage=lambda df: df["count"] / df["count"].sum() * 100
    )
)

Stage 6: Reporting

In [157]:
runs_per_match_report = (
    runs_per_match
    .sort_values(by="match_runs", ascending=False)
    .rename(columns={
        "match_id": "match_id",
        "match_runs": "total_runs"
    })
)

In [138]:
team_runs_per_match_report = (
    team_runs_per_match
    .sort_values(by=["match_id", "team_runs"], ascending=[True, False])
    .rename(columns={
        "team": "team_name",
        "team_runs": "runs_scored"
    })
)

In [165]:
top_10_batters = (
    batter_total_runs
    .sort_values(by="total_runs_by_batter", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_batters_report = (
    top_10_batters
    .rename(columns={
        "batter": "player",
        "total_runs": "runs"
    })
)

In [140]:
strike_rate_report = (
    batting_sr_df
    .sort_values(by="strike_rate", ascending=False)
    .rename(columns={
        "batter": "player"
    })
)


In [142]:
economy_report = (
    top_10_economy_bowlers
    .sort_values(by="economy")
    .rename(columns={
        "bowler": "player"
    })
)


In [143]:
consistent_batters_report = (
    most_consistent_batters
    .rename(columns={
        "batter": "player"
    })
)

In [144]:
boundary_report = (
    top_boundary_players
    .rename(columns={
        "batter": "player",
        "boundary_count": "total_boundaries"
    })
)

In [145]:
boundary_percentage_report = (
    top_boundary_percentage_batters
    .rename(columns={
        "batter": "player"
    })
)

In [146]:
dot_ball_report = (
    top_dot_ball_bowlers
    .rename(columns={
        "bowler": "player"
    })
)

In [147]:
powerplay_report = (
    top_powerplay_teams
    .rename(columns={
        "batting_team": "team"
    })
)

In [148]:
death_over_team_report = (
    top_death_over_teams
    .rename(columns={
        "batting_team": "team"
    })
)

death_over_batter_report = (
    top_death_over_batters
    .rename(columns={
        "batter": "player"
    })
)

In [149]:
venue_report = (
    venue_analysis
    .sort_values(by="average_match_runs", ascending=False)
)

city_report = (
    city_run_trends
    .sort_values(by="average_match_runs", ascending=False)
)

In [150]:
season_report = (
    season_run_trends
    .sort_values(by="season")
)

In [151]:
toss_impact_report = (
    toss_impact_summary
    .rename(columns={
        "team_type": "team_category"
    })
)

In [152]:
winning_accuracy_report = (
    winner_match_summary
    .assign(
        percentage=lambda df: df["count"] / df["count"].sum() * 100
    )
)

Stage 7: Data Export

In [166]:
import os

os.makedirs("output", exist_ok=True)


In [167]:
runs_per_match_report.to_csv(
    "output/runs_per_match.csv", index=False
)

team_runs_per_match_report.to_csv(
    "output/team_scores.csv", index=False
)

top_batters_report.to_csv(
    "output/top_batters.csv", index=False
)

strike_rate_report.to_csv(
    "output/strike_rate.csv", index=False
)

economy_report.to_csv(
    "output/economy.csv", index=False
)

consistent_batters_report.to_csv(
    "output/consistent_batters.csv", index=False
)

boundary_report.to_csv(
    "output/top_boundary_players.csv", index=False
)

boundary_percentage_report.to_csv(
    "output/boundary_percentage.csv", index=False
)

dot_ball_report.to_csv(
    "output/dot_ball_bowlers.csv", index=False
)

powerplay_report.to_csv(
    "output/powerplay_teams.csv", index=False
)

death_over_team_report.to_csv(
    "output/death_over_teams.csv", index=False
)

death_over_batter_report.to_csv(
    "output/death_over_batters.csv", index=False
)

venue_report.to_csv(
    "output/venue_analysis.csv", index=False
)

city_report.to_csv(
    "output/city_analysis.csv", index=False
)

season_report.to_csv(
    "output/season_trends.csv", index=False
)

toss_impact_report.to_csv(
    "output/toss_impact.csv", index=False
)

winning_accuracy_report.to_csv(
    "output/winning_accuracy.csv", index=False
)

In [171]:
with pd.ExcelWriter("ipl_analysis.xlsx", engine="openpyxl") as writer:

    runs_per_match_report.to_excel(
        writer, sheet_name="Runs per Match", index=False
    )

    team_runs_per_match_report.to_excel(
        writer, sheet_name="Team Scores", index=False
    )

    top_batters_report.to_excel(
        writer, sheet_name="Top Batters", index=False
    )

    strike_rate_report.to_excel(
        writer, sheet_name="Strike Rate", index=False
    )

    economy_report.to_excel(
        writer, sheet_name="Economy", index=False
    )

    consistent_batters_report.to_excel(
        writer, sheet_name="Consistent Batters", index=False
    )

    boundary_report.to_excel(
        writer, sheet_name="Boundaries", index=False
    )

    boundary_percentage_report.to_excel(
        writer, sheet_name="Boundary %", index=False
    )

    dot_ball_report.to_excel(
        writer, sheet_name="Dot Balls", index=False
    )

    powerplay_report.to_excel(
        writer, sheet_name="Powerplay", index=False
    )

    death_over_team_report.to_excel(
        writer, sheet_name="Death Overs Teams", index=False
    )

    death_over_batter_report.to_excel(
        writer, sheet_name="Death Overs Batters", index=False
    )

    venue_report.to_excel(
        writer, sheet_name="Venues", index=False
    )

    city_report.to_excel(
        writer, sheet_name="Cities", index=False
    )

    season_report.to_excel(
        writer, sheet_name="Season Trends", index=False
    )

    toss_impact_report.to_excel(
        writer, sheet_name="Toss Impact", index=False
    )

    winning_accuracy_report.to_excel(
        writer, sheet_name="Winning Accuracy", index=False
    )